
# CS 650 — Assignment #1: Relational Databases and Schema Design Using SQLite

### 📘 Overview
This assignment includes two major components:  
1. **🧪 Guided Lab** — A hands-on exercise to help you design, implement, and explore a relational schema using SQLite.  
2. **📝 Assignment Tasks** — A set of problems to complete independently after finishing the lab.

The lab introduces key database design and SQL concepts step-by-step, while the assignment section assesses your understanding through applied problem-solving and reasoning.

---

### 💡 Learning Objectives
By the end of this assignment, you will be able to:
- Design normalized **relational schemas** using primary and foreign keys.  
- Implement and enforce **data integrity constraints** such as `UNIQUE`, `CHECK`, and `NOT NULL`.  
- Create, populate, and query relational tables using **SQLite**.  
- Apply **JOIN** operations and **aggregations** to analyze multi-table data.  
- Understand and demonstrate **normalization** (1NF → 3NF) and **schema evolution**.  
- Analyze **query performance** using indexing and explain query plans.  

---

### 🧾 Domain Description
You will design and query a **Retail Transactions Database** that manages customers, orders, and products.

**Core Tables:**
- `customers(customer_id, name, email, city)`  
- `products(product_id, product_name, category, price)`  
- `orders(order_id, customer_id, order_date)`  
- `order_items(order_item_id, order_id, product_id, quantity, unit_price)`

**Relationships:**
- One **customer** → many **orders**  
- One **order** → many **order items**  
- Each **order item** → one **product**

This structure demonstrates how relational databases model hierarchical and transactional data efficiently.



---

# 🧪 LAB (Guided)

In this guided lab, you will design, implement, and query a relational database schema using SQLite. Each step will walk you through key database concepts—such as table creation, normalization, and indexing—so you can apply them hands-on. By completing the lab, you will build both the practical skills and conceptual understanding needed for the independent assignment tasks that follow.



### 🔁 Lab 0 — Reset the environment
This optional reset removes the existing SQLite file so you can re-run the lab from a clean slate.


- It performs a **step** in the end-to-end pipeline (setup, query, or validation).
- Keep the output and reflect on how it links to the schema design and normalization goals.

This block continues the workflow, connecting theoretical concepts of normalization with practical SQL implementation.

In [ ]:

import os, sqlite3, pathlib

DB_PATH = "relational_lab.db"
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)
print("Reset complete. Fresh start!")


Reset complete. Fresh start!



### ⚙️ Lab 1 — Connect to SQLite & enable foreign keys
SQLite is file-based. We connect to `relational_lab.db` and enable foreign keys explicitly.


- It **enables foreign key enforcement** in SQLite so referential integrity rules actually run.
- Without this, deletes/inserts might silently violate relationships.

Foreign key enforcement is enabled to ensure that referential integrity rules are active during inserts and updates.

In [ ]:

import sqlite3
conn = sqlite3.connect("relational_lab.db")
conn.execute("PRAGMA foreign_keys = ON;")
print("Connected. Foreign keys enabled:", conn.execute("PRAGMA foreign_keys;").fetchone()[0])


Connected. Foreign keys enabled: 1



### 🧱 Lab 2 — Create schema with keys & constraints
We define four tables with **PRIMARY KEY**, **FOREIGN KEY**, **UNIQUE**, and **CHECK** constraints.

- `customers.email` is **UNIQUE** (no duplicates)
- `products.price` must be non-negative (**CHECK(price >= 0)**)
- `order_items.quantity` must be positive
- `order_items.unit_price` captures price at the time of order (denormalization for history)


- It **defines the schema** (tables, columns, data types) that enforces structure.
- **Primary/foreign keys** encode relationships and prevent inconsistent data.
- Constraints (e.g., `UNIQUE`, `CHECK`, `NOT NULL`) support **data quality** and normalization.

This block defines the **relational schema**, specifying tables, columns, data types, and keys. It establishes the structural foundation for normalization and ensures that relationships and constraints are clearly enforced.

In [ ]:

schema_sql = """
CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY,
    name        TEXT NOT NULL,
    email       TEXT NOT NULL UNIQUE,
    city        TEXT
);

CREATE TABLE products (
    product_id   INTEGER PRIMARY KEY,
    product_name TEXT NOT NULL,
    category     TEXT,
    price        REAL NOT NULL CHECK (price >= 0)
);

CREATE TABLE orders (
    order_id    INTEGER PRIMARY KEY,
    customer_id INTEGER NOT NULL,
    order_date  TEXT NOT NULL,
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
);

CREATE TABLE order_items (
    order_item_id INTEGER PRIMARY KEY,
    order_id      INTEGER NOT NULL,
    product_id    INTEGER NOT NULL,
    quantity      INTEGER NOT NULL CHECK (quantity > 0),
    unit_price    REAL NOT NULL CHECK (unit_price >= 0),
    FOREIGN KEY (order_id)  REFERENCES orders(order_id),
    FOREIGN KEY (product_id) REFERENCES products(product_id)
);
"""
conn.executescript(schema_sql)
conn.commit()
print("Schema created.")


Schema created.



### 🧪 Lab 3 — Insert sample data
We insert a few rows in each table to support join queries and aggregations.


- It **loads seed data** so you can test joins, constraints, and queries.
- Test data enables you to validate normalization and integrity assumptions.

This section populates the tables with representative data, allowing verification of normalization and constraint behavior through real examples.

In [ ]:

conn.executescript("""
INSERT INTO customers (customer_id, name, email, city) VALUES
  (1, 'Alice Chen', 'alice@example.com', 'Norfolk'),
  (2, 'Brian Diaz', 'brian@example.com', 'Virginia Beach'),
  (3, 'Carla Singh', 'carla@example.com', 'Chesapeake');

INSERT INTO products (product_id, product_name, category, price) VALUES
  (101, 'Wireless Mouse', 'Accessories', 24.99),
  (102, 'Mechanical Keyboard', 'Accessories', 99.50),
  (103, 'USB-C Hub', 'Accessories', 34.00),
  (201, '27-inch Monitor', 'Displays', 229.99),
  (202, 'Laptop Stand', 'Accessories', 39.99);

INSERT INTO orders (order_id, customer_id, order_date) VALUES
  (1001, 1, '2025-09-21'),
  (1002, 1, '2025-10-02'),
  (1003, 2, '2025-10-05');

INSERT INTO order_items (order_item_id, order_id, product_id, quantity, unit_price) VALUES
  (1, 1001, 101, 1, 24.99),
  (2, 1001, 201, 1, 229.99),
  (3, 1002, 102, 1, 99.50),
  (4, 1002, 103, 2, 34.00),
  (5, 1003, 202, 1, 39.99);
""")
conn.commit()
print("Sample data inserted.")


Sample data inserted.



### 🔎 Lab 4 — Basic SELECTs with filters & ordering
We’ll practice simple queries on single tables.


- It performs a **step** in the end-to-end pipeline (setup, query, or validation).
- Keep the output and reflect on how it links to the schema design and normalization goals.

This block continues the workflow, connecting theoretical concepts of normalization with practical SQL implementation.

In [ ]:

def q(sql):
    cur = conn.execute(sql)
    cols = [d[0] for d in cur.description]
    rows = cur.fetchall()
    from pandas import DataFrame
    df = DataFrame(rows, columns=cols)
    display(df)

q("SELECT * FROM products ORDER BY price DESC;")
q("SELECT name, email FROM customers WHERE city LIKE 'V%';")


,product_id,product_name,category,price
0,201,27-inch Monitor,Displays,229.99
1,102,Mechanical Keyboard,Accessories,99.50
2,202,Laptop Stand,Accessories,39.99
3,103,USB-C Hub,Accessories,34.00
4,101,Wireless Mouse,Accessories,24.99


,name,email
0,Brian Diaz,brian@example.com



### 🔗 Lab 5 — Join queries (one-to-many & many-to-one)
We join across `orders`, `order_items`, `products`, and `customers`.


- It demonstrates **joining normalized tables** to reconstruct business views.
- Shows how **keys** and **foreign keys** enable combining related entities.

These queries demonstrate how normalized tables can be joined to reconstruct meaningful views of data while maintaining integrity across relations.

In [ ]:

q("""
SELECT o.order_id, o.order_date, c.name AS customer,
       p.product_name, oi.quantity, oi.unit_price,
       (oi.quantity * oi.unit_price) AS line_total
FROM orders o
JOIN customers c ON c.customer_id = o.customer_id
JOIN order_items oi ON oi.order_id = o.order_id
JOIN products p ON p.product_id = oi.product_id
ORDER BY o.order_id, p.product_name;
""")

q("""
-- Order totals by order_id
SELECT o.order_id,
       ROUND(SUM(oi.quantity * oi.unit_price), 2) AS order_total
FROM orders o
JOIN order_items oi ON oi.order_id = o.order_id
GROUP BY o.order_id
ORDER BY o.order_id;
""")


,order_id,order_date,customer,product_name,quantity,unit_price,line_total
0,1001,2025-09-21,Alice Chen,27-inch Monitor,1,229.99,229.99
1,1001,2025-09-21,Alice Chen,Wireless Mouse,1,24.99,24.99
2,1002,2025-10-02,Alice Chen,Mechanical Keyboard,1,99.50,99.50
3,1002,2025-10-02,Alice Chen,USB-C Hub,2,34.00,68.00
4,1003,2025-10-05,Brian Diaz,Laptop Stand,1,39.99,39.99


,order_id,order_total
0,1001,254.98
1,1002,167.50
2,1003,39.99



### 🛡️ Lab 6 — Constraint & integrity demos
We try to violate a **UNIQUE** and a **FOREIGN KEY** to see proper errors.


- It **loads seed data** so you can test joins, constraints, and queries.
- Test data enables you to validate normalization and integrity assumptions.

This section populates the tables with representative data, allowing verification of normalization and constraint behavior through real examples.

In [ ]:

# UNIQUE violation (duplicate email)
try:
    conn.execute("INSERT INTO customers (name, email, city) VALUES ('Julia', 'alice@example.com', 'Norfolk');")
    conn.commit()
except Exception as e:
    print("Expected UNIQUE violation:", e)

# FOREIGN KEY violation (order references missing customer)
try:
    conn.execute("INSERT INTO orders (order_id, customer_id, order_date) VALUES (2000, 999, '2025-10-10');")
    conn.commit()
except Exception as e:
    print("Expected FK violation:", e)


Expected UNIQUE violation: UNIQUE constraint failed: customers.email
Expected FK violation: FOREIGN KEY constraint failed



### 🧩 Lab 7 — Normalization walkthrough (1NF → 3NF) _explanation_
**Problem:** Suppose we stored `orders` like: `(order_id, customer_name, customer_email, products_csv, total)` with multiple product names in one field.

- **1NF:** No repeating groups. Split out **order_items** so every row is one product for one order.
- **2NF:** Remove partial dependencies. Product details belong in **products**, not in **order_items**.
- **3NF:** Remove transitive dependencies. Customer attributes (name, email, city) belong in **customers**.

This is what our current schema implements.



### 🔧 Lab 8 — Schema evolution (add a column, backfill, use it)
We add a `loyalty_tier` to `customers` and then run queries using it.


- It models **schema evolution**—changing requirements without losing data.
- Reinforces how good normalization localizes change and reduces ripple effects.

Here, the schema is evolved to accommodate new requirements, showing how a normalized design supports flexibility and scalability.

In [ ]:

conn.executescript("""
ALTER TABLE customers ADD COLUMN loyalty_tier TEXT DEFAULT 'Bronze';
UPDATE customers SET loyalty_tier = 'Gold' WHERE customer_id = 1;
UPDATE customers SET loyalty_tier = 'Silver' WHERE customer_id = 2;
""")
conn.commit()

q("SELECT customer_id, name, city, loyalty_tier FROM customers ORDER BY customer_id;")


,customer_id,name,city,loyalty_tier
0,1,Alice Chen,Norfolk,Gold
1,2,Brian Diaz,Virginia Beach,Silver
2,3,Carla Singh,Chesapeake,Bronze



### 🚀 Lab 9 — Indexing & Query Plans
We compare a query plan **before** and **after** adding an index on `products(category)`.


- It introduces **indexing**, which can dramatically reduce query cost on large tables.
- Connects to **performance** topics and reading **query plans**.

An index is created to optimize data retrieval performance, illustrating how normalization and indexing together enhance query efficiency.

In [ ]:

def explain(sql):
    print(sql.strip())
    for row in conn.execute("EXPLAIN QUERY PLAN " + sql):
        print("  ->", row)

print("Before index:")
explain("SELECT * FROM products WHERE category = 'Accessories';")

conn.execute("CREATE INDEX IF NOT EXISTS idx_products_category ON products(category);")
conn.commit()

print("\nAfter index:")
explain("SELECT * FROM products WHERE category = 'Accessories';")


Before index:
SELECT * FROM products WHERE category = 'Accessories';
  -> (2, 0, 0, 'SCAN products')

After index:
SELECT * FROM products WHERE category = 'Accessories';
  -> (3, 0, 0, 'SEARCH products USING INDEX idx_products_category (category=?)')



### 🪟 Lab 10 — (Optional) Views for convenience
We create a `order_totals_view` to simplify reporting.


- It demonstrates **joining normalized tables** to reconstruct business views.
- Shows how **keys** and **foreign keys** enable combining related entities.

These queries demonstrate how normalized tables can be joined to reconstruct meaningful views of data while maintaining integrity across relations.

In [ ]:

conn.executescript("""
CREATE VIEW IF NOT EXISTS order_totals_view AS
SELECT o.order_id,
       c.name AS customer,
       o.order_date,
       ROUND(SUM(oi.quantity * oi.unit_price), 2) AS order_total
FROM orders o
JOIN customers c ON c.customer_id = o.customer_id
JOIN order_items oi ON oi.order_id = o.order_id
GROUP BY o.order_id, c.name, o.order_date;
""")
q("SELECT * FROM order_totals_view ORDER BY order_id;")


,order_id,customer,order_date,order_total
0,1001,Alice Chen,2025-09-21,254.98
1,1002,Alice Chen,2025-10-02,167.50
2,1003,Brian Diaz,2025-10-05,39.99



---

# 📝 ASSIGNMENT TASKS (Student Work)

**Instructions:** Answer in separate cells under each task. Use **SQL + short written justifications** (2–4 sentences).

### ✅ Task 1 — Extend the schema with Addresses
Customers may have **multiple shipping addresses**. Design a new table `addresses` and relate it to `customers`.
- Show `CREATE TABLE` with **PK**, **FK**, and at least one **CHECK**.
- Insert 2–3 example rows for Alice and Brian.
- Query: list all customers with all addresses (1 row per address).

> _Add your cells below:_


In [ ]:
conn.executescript("""
CREATE TABLE IF NOT EXISTS addresses (
    address_id INTEGER PRIMARY KEY,
    customer_id INTEGER NOT NULL,
    street TEXT NOT NULL,
    city TEXT NOT NULL,
    state TEXT NOT NULL,
    zip_code TEXT NOT NULL,
    address_type TEXT NOT NULL CHECK (address_type IN ('Home', 'Work')),
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
);
""")

conn.commit()
print("Addresses table created.")

Addresses table created.


In [ ]:
conn.executescript("""
INSERT INTO addresses
(address_id, customer_id, street, city, state, zip_code, address_type)
VALUES
    (1, 1, '120 Main Street', 'Norfolk', 'VA', '23510', 'Home'),
    (2, 1, '400 Granby Street', 'Norfolk', 'VA', '23510', 'Work'),
    (3, 2, '800 Atlantic Avenue', 'Virginia Beach', 'VA', '23451', 'Home');
""")

conn.commit()
print("Example addresses inserted.")

Example addresses inserted.


In [ ]:
q("""
SELECT
    c.customer_id,
    c.name,
    a.street,
    a.city,
    a.state,
    a.zip_code,
    a.address_type
FROM customers c
JOIN addresses a
    ON c.customer_id = a.customer_id
ORDER BY c.customer_id, a.address_id;
""")

,customer_id,name,street,city,state,zip_code,address_type
0,1,Alice Chen,120 Main Street,Norfolk,VA,23510,Home
1,1,Alice Chen,400 Granby Street,Norfolk,VA,23510,Work
2,2,Brian Diaz,800 Atlantic Avenue,Virginia Beach,VA,23451,Home


I created the addresses table so that each customer can have multiple shipping addresses. The customer_id foreign key connects each address to a customer, while the CHECK constraint limits address_type to Home or Work. This keeps address data separate from the customers table and supports the one-to-many relationship.


---


### ✅ Task 2 — Business Query: Top-Spending Customers
Compute the **top 2** customers by **total spend** across all orders.
- Show the SQL query (joins + aggregation).
- Show the result table.
- Briefly explain how ties would be handled and how to include a **date range** filter.

> _Add your cells below:_


In [ ]:
q("""
SELECT
    c.customer_id,
    c.name,
    ROUND(SUM(oi.quantity * oi.unit_price), 2) AS total_spend
FROM customers c
JOIN orders o
    ON c.customer_id = o.customer_id
JOIN order_items oi
    ON o.order_id = oi.order_id
GROUP BY c.customer_id, c.name
ORDER BY total_spend DESC
LIMIT 2;
""")

,customer_id,name,total_spend
0,1,Alice Chen,422.48
1,2,Brian Diaz,39.99


The query joins customers, orders, and order_items and calculates each customer's total spending using SUM(quantity * unit_price). LIMIT 2 returns only the two customers with the highest totals. If customers are tied, LIMIT 2 may exclude one of the tied customers; a ranking function such as DENSE_RANK could include all customers tied at the cutoff. A date range can be added with a WHERE condition on o.order_date before the GROUP BY.


---


### ✅ Task 3 — Data Quality & Constraints
1) Attempt to insert a product with a **negative price** and show that it fails.  
2) Update the schema to add a **UNIQUE(category, product_name)** constraint (or explain why/why not).  
3) Show a query that finds **orphan order_items** (should be none with FKs on).

> _Add your cells below:_


In [ ]:
try:
    conn.execute("""
        INSERT INTO products
        (product_id, product_name, category, price)
        VALUES (999, 'Test Product', 'Test', -10.00);
    """)
    conn.commit()
except Exception as e:
    print("Expected CHECK constraint failure:", e)
    conn.rollback()

Expected CHECK constraint failure: CHECK constraint failed: price >= 0


In [ ]:
conn.execute("""
CREATE UNIQUE INDEX IF NOT EXISTS
idx_products_category_product_name
ON products(category, product_name);
""")

conn.commit()

print("Unique constraint behavior added using a unique index.")

Unique constraint behavior added using a unique index.


In [ ]:
q("""
SELECT
    oi.order_item_id,
    oi.order_id,
    oi.product_id
FROM order_items oi
LEFT JOIN orders o
    ON oi.order_id = o.order_id
LEFT JOIN products p
    ON oi.product_id = p.product_id
WHERE o.order_id IS NULL
   OR p.product_id IS NULL;
""")

,order_item_id,order_id,product_id


The negative product price was rejected because the products table has a CHECK constraint requiring price to be at least zero. Since SQLite does not support adding a new table-level UNIQUE constraint to an existing table with a simple ALTER TABLE statement, I used a unique index on category and product_name to enforce the same rule. The orphan query returned no rows, which shows that the existing foreign key relationships are maintaining referential integrity.


---


### ✅ Task 4 — Indexing Experiment
- Create an index that could speed up **order totals by date range**.
- Show `EXPLAIN QUERY PLAN` before vs after your index.
- Briefly explain why the plan changed (or didn’t).

> _Add your cells below:_


In [ ]:
print("Before index:")

for row in conn.execute("""
EXPLAIN QUERY PLAN
SELECT *
FROM orders
WHERE order_date BETWEEN '2025-09-01' AND '2025-10-31';
"""):
    print(row)

Before index:
(2, 0, 0, 'SCAN orders')


In [ ]:
conn.execute("""
CREATE INDEX IF NOT EXISTS idx_orders_order_date
ON orders(order_date);
""")

conn.commit()

print("Index created on orders(order_date).")

Index created on orders(order_date).


In [ ]:
print("After index:")

for row in conn.execute("""
EXPLAIN QUERY PLAN
SELECT *
FROM orders
WHERE order_date BETWEEN '2025-09-01' AND '2025-10-31';
"""):
    print(row)

After index:
(3, 0, 0, 'SEARCH orders USING INDEX idx_orders_order_date (order_date>? AND order_date<?)')


I created an index on orders(order_date) to make date-range searches more efficient. Before adding the index, the query plan showed a scan of the orders table. After adding it, SQLite used the new index to search the orders by date. This would be more helpful as the orders table gets larger.


---


### ✅ Task 5 — Normalization Reasoning
Given a hypothetical `orders_wide(order_id, customer_email, product_1, product_2, product_3, ...)`:
- Identify **two** anomalies this design causes (update/insert/delete).
- Rewrite the design using **3NF**-friendly tables.
- Write one query that proves your design is usable for analytics.

> _Add your cells below:_


The orders_wide design can cause an update anomaly because the same customer email or product information could appear in multiple rows and would need to be updated everywhere. It can also cause an insert anomaly because an order with more products than the available product columns would require changing the table structure.

A 3NF-friendly design separates the information into customers(customer_id, name, email, city), products(product_id, product_name, category, price), orders(order_id, customer_id, order_date), and order_items(order_item_id, order_id, product_id, quantity, unit_price). This removes repeating product columns and stores each type of information in the table where it belongs.


In [ ]:
q("""
SELECT
    p.product_name,
    SUM(oi.quantity) AS total_quantity_sold,
    ROUND(SUM(oi.quantity * oi.unit_price), 2) AS total_sales
FROM products p
JOIN order_items oi
    ON p.product_id = oi.product_id
GROUP BY p.product_id, p.product_name
ORDER BY total_sales DESC;
""")

,product_name,total_quantity_sold,total_sales
0,27-inch Monitor,1,229.99
1,Mechanical Keyboard,1,99.50
2,USB-C Hub,2,68.00
3,Laptop Stand,1,39.99
4,Wireless Mouse,1,24.99



---

# 📤 Submission & 📏 Grading

### What to Submit
- This completed **notebook (.ipynb)** exported to **PDF** (or both `.ipynb` + PDF if allowed by Canvas).
- Include your **SQL outputs** and **short explanations** under each task.
- Name your files using: `CS660_Assignment1_Lastname_Firstname.ipynb` (+ `.pdf`).

### Rubric (Completion-Oriented)
- ✅ Lab cells executed without errors (foreign keys on, schema created, joins work). **(30%)**
- ✅ Tasks 1–5 attempted with reasonable SQL and explanations. **(60%)**
- ✅ Clear formatting, headings, and submission naming. **(10%)**

> **Academic Integrity:** You may discuss ideas with peers, but all code and explanations must be your own. Cite any sources used.

---

## 🙋 Troubleshooting Tips
- **Foreign keys not enforced?** Make sure `PRAGMA foreign_keys = ON;` runs after _every_ new connection.
- **Reset and re-run:** Use the **Reset** cell at the top to recreate a clean database file.
- **Where is my DB file?** It’s `relational_lab.db` in the current working directory.



---

## ✅ (Optional) Close Connection
If you ran everything above and are done, close the connection.


In [ ]:

try:
    conn.close()
    print("Connection closed.")
except:
    pass
